# Domain 4 — Eval, Testing, and Debugging (2.6%)

One skill: Debugging and Error Handling. The core competency is **isolating
the problem origin** — did the integration layer fail, or did the model
produce a bad answer from correct inputs? They need opposite fixes, and
confusing them is how teams waste days.

| Symptom | Origin | Fix |
|---|---|---|
| `TypeError` unpacking `tool_use.input` | Integration | Align your function signature with the schema |
| Model cites a policy clause that does not exist | Model output | Ground it with retrieval; validate against source |
| `tool_result` missing for a `tool_use` | Integration | Answer every tool_use in the next turn |
| Model picks the wrong tool from clear options | Model output | Improve tool descriptions |
| Retry loop repeats the same malformed JSON | Integration | Feed the error back so it can self-correct |


In [ ]:
"""Shared setup. Export ANTHROPIC_API_KEY before launching Jupyter."""

import json
import os

import anthropic

client = anthropic.Anthropic()

OPUS = "claude-opus-5"
SONNET = "claude-sonnet-5"
HAIKU = "claude-haiku-4-5-20251001"

MODEL = SONNET


def extract_text(response: anthropic.types.Message) -> str:
    """Concatenate text blocks, ignoring thinking and tool_use blocks."""
    return "".join(
        block.text for block in response.content if block.type == "text"
    )


print("API key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))


## 4.1 Instrumented traces

You cannot debug what you cannot see. Record every turn: stop reason, tool
calls, token usage, and errors. This trace is also what produces the
latency and cost numbers you need for Domain 5.


In [ ]:
import time
from dataclasses import dataclass, field


@dataclass
class TurnTrace:
    """One turn of an agent loop."""

    turn: int
    stop_reason: str
    tool_calls: list[str]
    input_tokens: int
    output_tokens: int
    elapsed_seconds: float
    errors: list[str] = field(default_factory=list)


@dataclass
class RunTrace:
    """A whole agent run, with roll-up totals."""

    goal: str
    turns: list[TurnTrace] = field(default_factory=list)
    outcome: str = "incomplete"

    def total_tokens(self) -> tuple[int, int]:
        """Return (input, output) totals across all turns."""
        return (
            sum(turn.input_tokens for turn in self.turns),
            sum(turn.output_tokens for turn in self.turns),
        )

    def report(self) -> None:
        """Print the trace in a form you can actually read."""
        print(f"GOAL    : {self.goal}")
        print(f"OUTCOME : {self.outcome}")
        for turn in self.turns:
            errors = f"  ERRORS={turn.errors}" if turn.errors else ""
            print(
                f"  turn {turn.turn}: stop={turn.stop_reason:<10} "
                f"tools={turn.tool_calls} "
                f"tok={turn.input_tokens}/{turn.output_tokens} "
                f"{turn.elapsed_seconds:.2f}s{errors}"
            )
        tokens_in, tokens_out = self.total_tokens()
        print(f"  TOTAL: {tokens_in} in / {tokens_out} out "
              f"across {len(self.turns)} turns")


In [ ]:
CLAIMS = {"ABC-123": {"status": "APPROVED", "amount": 8400}}

TOOLS = [
    {
        "name": "get_claim",
        "description": (
            "Retrieve a claim record by its exact ID (format ABC-123). "
            "Returns status and payout amount."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "claim_id": {"type": "string", "pattern": "^[A-Z]{3}-[0-9]{3}$"}
            },
            "required": ["claim_id"],
        },
    }
]


def get_claim(claim_id: str) -> str:
    """Return a claim record, or an explicit error string."""
    record = CLAIMS.get(claim_id)
    if record is None:
        return f"ERROR: no claim with id {claim_id}"
    return json.dumps(record)


def run_traced_agent(goal: str, max_turns: int = 5) -> RunTrace:
    """Run a bounded agent loop, recording a trace of every turn."""
    trace = RunTrace(goal=goal)
    messages: list[dict] = [{"role": "user", "content": goal}]

    for turn_number in range(1, max_turns + 1):
        start = time.perf_counter()
        response = client.messages.create(
            model=MODEL, max_tokens=1024, tools=TOOLS, messages=messages
        )
        elapsed = time.perf_counter() - start
        messages.append({"role": "assistant", "content": response.content})

        tool_calls = [
            block for block in response.content if block.type == "tool_use"
        ]
        turn = TurnTrace(
            turn=turn_number,
            stop_reason=response.stop_reason,
            tool_calls=[call.name for call in tool_calls],
            input_tokens=response.usage.input_tokens,
            output_tokens=response.usage.output_tokens,
            elapsed_seconds=elapsed,
        )

        if response.stop_reason != "tool_use":
            trace.turns.append(turn)
            trace.outcome = extract_text(response)[:120]
            return trace

        results = []
        for call in tool_calls:
            try:
                output = get_claim(**call.input)
                if output.startswith("ERROR:"):
                    turn.errors.append(output)
            except TypeError as error:
                # Schema and signature disagree: an INTEGRATION bug.
                output = f"ERROR: bad arguments: {error}"
                turn.errors.append(output)

            results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": call.id,
                    "content": output,
                    "is_error": output.startswith("ERROR:"),
                }
            )

        trace.turns.append(turn)
        messages.append({"role": "user", "content": results})

    trace.outcome = f"STOPPED: hit the {max_turns}-turn limit"
    return trace


run_traced_agent("What is the status of claim ABC-123?").report()
print()
run_traced_agent("What is the status of claim QQQ-999?").report()


## 4.2 Testing non-deterministic output

You cannot assert exact string equality on model output — it varies run to
run. Assert on **properties** instead: does it validate against the schema,
is it in the allowed set, does it agree with ground truth?

And run each case several times. A single green run on a stochastic system
tells you almost nothing; a pass rate does.


In [ ]:
from pydantic import BaseModel, Field, ValidationError


class Verdict(BaseModel):
    """Expected output contract."""

    decision: str = Field(pattern=r"^(APPROVE|DENY|REVIEW)$")
    reason: str = Field(min_length=5, max_length=200)


def get_verdict(claim_text: str) -> str:
    """Return the raw JSON string the model produced."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        system=(
            "Return JSON only: {\"decision\": APPROVE|DENY|REVIEW, "
            "\"reason\": short string}."
        ),
        messages=[
            {"role": "user", "content": claim_text},
            {"role": "assistant", "content": "{"},
        ],
    )
    return "{" + extract_text(response)


def evaluate(cases: list[tuple[str, str]], runs: int = 3) -> None:
    """Score structural validity and agreement with expected decisions.

    Two separate metrics on purpose: valid-but-wrong and invalid are
    different failures needing different fixes.
    """
    valid_count = 0
    correct_count = 0
    total = 0

    for claim_text, expected in cases:
        decisions = []
        for _ in range(runs):
            total += 1
            raw = get_verdict(claim_text)
            try:
                verdict = Verdict.model_validate_json(raw)
            except (ValidationError, json.JSONDecodeError):
                decisions.append("INVALID")
                continue

            valid_count += 1
            decisions.append(verdict.decision)
            if verdict.decision == expected:
                correct_count += 1

        agreement = decisions.count(expected) / runs
        print(f"{expected:8} expected | got {decisions} | "
              f"agreement {agreement:.0%}")

    print(f"\nschema-valid : {valid_count}/{total} "
          f"({valid_count / total:.0%})")
    print(f"correct      : {correct_count}/{total} "
          f"({correct_count / total:.0%})")


evaluate(
    [
        ("Kitchen fire, fire peril covered, $22,000 damage.", "APPROVE"),
        ("Gradual roof wear from age; wear is explicitly excluded.", "DENY"),
        ("Cause of floor damage undetermined; no inspection yet.", "REVIEW"),
    ]
)


## 4.3 Isolating the origin — worked cases

Run the diagnostic below, then reason through each case yourself before
reading the verdict.


In [ ]:
def diagnose(symptom: str, origin: str, fix: str) -> None:
    """Print a structured diagnosis of one failure."""
    print(f"SYMPTOM : {symptom}")
    print(f"ORIGIN  : {origin}")
    print(f"FIX     : {fix}\n")


diagnose(
    "TypeError: get_claim() got an unexpected keyword argument 'id'",
    "Integration layer",
    "The schema advertises 'id' but the function takes 'claim_id'. The "
    "model did exactly what the schema said. Align the two.",
)

diagnose(
    "Model reports APPROVED when the tool_result clearly said DENIED",
    "Model output",
    "Inputs were correct; reasoning was not. Strengthen the prompt, add "
    "a validator that cross-checks the answer against the tool_result, "
    "and consider a stronger tier if it persists.",
)

diagnose(
    "400 error: tool_use ids in the assistant turn have no tool_result",
    "Integration layer",
    "Every tool_use block needs a matching tool_result in the very next "
    "user message. You are almost certainly handling only the first of "
    "several parallel calls.",
)

diagnose(
    "Retry loop makes three identical calls and fails identically",
    "Integration layer",
    "The retry resends the original prompt without telling the model what "
    "was wrong. Append the validation error to the conversation so the "
    "next attempt has new information.",
)

diagnose(
    "Agent runs 40 turns and never finishes",
    "Integration layer (missing guardrail)",
    "No turn limit. Bound the loop, and inspect the trace for the turn "
    "where progress stopped -- usually a tool returning something the "
    "model cannot act on.",
)
